In [ ]:
# CÉLULA 1: INSTALAÇÃO

!pip install coqui-tts stable-ts pyphen pydub openai

In [ ]:
import os
from google.colab import files
import shutil

# Limpa qualquer resultado de uma execução anterior
if os.path.exists('output'):
    shutil.rmtree('output')
if os.path.exists('uploaded_audio.tmp'):
    os.remove('uploaded_audio.tmp')

# Abre a janela para o usuário escolher o arquivo
uploaded = files.upload()

# Pega o nome do arquivo que o usuário enviou
input_filename = list(uploaded.keys())[0]

# Renomeia temporariamente para um nome padrão para facilitar os próximos passos
os.rename(input_filename, 'uploaded_audio.tmp')

# Pega o nome do arquivo sem a extensão (ex: "minha_musica")
# Isso é crucial para encontrar a pasta de saída correta do Demucs
base_name = "uploaded_audio"

print(f"\nArquivo '{input_filename}' carregado com sucesso!")
print("Pronto para separar as faixas na próxima célula.")

In [3]:
import requests
import os
import time

API_KEY = "03ee442b-17c3-43aa-bd2c-09753e410fa2"

headers = {
    "Authorization": API_KEY
}

res = requests.get("https://api.music.ai/v1/upload", headers=headers)
upload_info = res.json()

upload_url = upload_info['uploadUrl']
download_url = upload_info['downloadUrl']


In [ ]:
# Caminho local do arquivo
file_path = "uploaded_audio.tmp"

with open(file_path, "rb") as f:
    put_res = requests.put(upload_url, data=f)
    print("Status do upload:", put_res.status_code)


In [ ]:
payload = {
    "name": "Separar Stems",
    "workflow": "separador_main",
    "params": {
        "inputUrl": download_url
    }
}

job_res = requests.post(
    "https://api.music.ai/v1/job",
    headers={**headers, "Content-Type": "application/json"},
    json=payload
)

print(job_res.json())

response_data = job_res.json()
job_id = response_data['id']

In [ ]:
for i in range(2):
  res = requests.get(f"https://api.music.ai/v1/job/{job_id}", headers=headers)
  time.sleep(30)
  data = res.json()
  output_dir = "/content/output"
  os.makedirs(output_dir, exist_ok=True)
  for stem, url in data["result"].items():
        print(f"{stem}: {url}")
        local_path = f"{output_dir}/{stem}.wav"
        print(f"Baixando {stem} → {local_path}")
        with requests.get(url, stream=True) as r:
          r.raise_for_status()
          with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print()

In [ ]:
#CÉLULA 3: MARCAÇÃO DOS TEMPOS E EXTRAÇÃO DA LETRA
import stable_whisper

caminho_vocal_original = '/content/output/Letra.wav'

print("Carregando o modelo Whisper... (Isso pode levar um momento)")
model = stable_whisper.load_model('medium') # 'large-v3' é o mais preciso

print(f"Transcrevendo '{caminho_vocal_original}' para obter o mapa de tempo...")
resultado_whisper = model.transcribe(caminho_vocal_original, language='en', regroup=True) # Mude 'en' se a língua original for outra

# Extrai os segmentos (frases) com seus tempos
segmentos_originais = resultado_whisper.segments
print(f"\nExtração de tempo concluída! {len(segmentos_originais)} segmentos de áudio encontrados.")

# Exemplo de como os dados ficam:
print("Exemplo dos 3 primeiros segmentos e seus tempos:")
for i, seg in enumerate(segmentos_originais[:3]):
    print(f"  Segmento {i}: '{seg.text.strip()}' -> Início: {seg.start:.2f}s, Fim: {seg.end:.2f}s")

In [8]:
import os
from openai import OpenAI
import time

from google.colab import userdata
OPENAI_API_KEY = userdata.get('openai_apikey')
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
letra_completa_numerada = ""
for i, seg in enumerate(segmentos_originais):
    letra_completa_numerada += f"{i}: {seg.text.strip()}\n"

print("Letra original formatada para envio à OpenAI:")
print(letra_completa_numerada)

In [ ]:
prompt_sistema = """
Você é um tradutor especialista em letras de música, traduzindo do Inglês para o Português do Brasil.
Sua tarefa é traduzir a letra a seguir.

Regras importantes:
1.  **Contexto é tudo**: Não traduza literalmente. Capture a emoção, a poesia e o significado da música.
2.  **Métrica e Ritmo (Regra Adicionada)**: Este é um ponto crucial. Esforce-se para que a contagem de sílabas fonéticas de cada verso em português seja o mais próxima possível da contagem do verso original em inglês. O objetivo é criar uma versão que possa ser cantada, mantendo o fluxo e a cadência da melodia original.
3.  **Consistência no Refrão**: Versos que se repetem (como refrões ou pontes) DEVEM ser traduzidos exatamente da mesma forma todas as vezes que aparecerem.
4.  **Formato de Saída**: A letra está numerada. Retorne a tradução mantendo EXATAMENTE a mesma numeração linha por linha. Não adicione ou remova linhas. O formato deve ser:
    NÚMERO: Tradução da linha

Exemplo de saída esperada:
0: Tradução da primeira linha
1: Tradução da segunda linha
2: Tradução da terceira linha
...
"""

segmentos_traduzidos = []
print("\nIniciando a tradução com a OpenAI... (Isso pode levar um momento)")

try:
    # Fazemos uma única chamada com a letra inteira
    resposta_openai = client.chat.completions.create(
        model="gpt-4o", # "gpt-4o" é excelente para nuance. "gpt-3.5-turbo" é uma opção mais rápida e barata.
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": letra_completa_numerada}
        ],
        temperature=0.3 # Uma temperatura mais baixa para manter a consistência
    )

    # Extrai o texto da resposta
    traducao_bruta = resposta_openai.choices[0].message.content

    print("\nTradução recebida da OpenAI. Processando o resultado...")
    # Cria um dicionário para mapear o número da linha à sua tradução
    traducoes_mapeadas = {}
    for linha in traducao_bruta.strip().split('\n'):
        partes = linha.split(':', 1)
        if len(partes) == 2:
            try:
                numero = int(partes[0])
                texto_traduzido = partes[1].strip()
                traducoes_mapeadas[numero] = texto_traduzido
            except ValueError:
                print(f"Aviso: Ignorando linha mal formatada da IA: '{linha}'")
                continue

    # Agora, montamos a lista final de segmentos traduzidos usando o mapa
    for i, seg in enumerate(segmentos_originais):
        texto_original = seg.text.strip()
        # Busca a tradução no mapa. Se não encontrar, usa o original como fallback.
        texto_traduzido = traducoes_mapeadas.get(i, texto_original)

        segmentos_traduzidos.append({
            "inicio": seg.start,
            "fim": seg.end,
            "texto_original": texto_original,
            "texto_traduzido": texto_traduzido
        })
        print(f"  Segmento {i}: '{texto_original}' -> '{texto_traduzido}'")

    print("\nTradução com OpenAI concluída com sucesso!")

except Exception as e:
    print(f"\nOcorreu um erro ao chamar a API da OpenAI: {e}")
    print("A lista 'segmentos_traduzidos' pode estar vazia ou incompleta.")

In [ ]:
# CÉLULA 5: CARREGAR O MODELO TTS

import torch
from TTS.api import TTS

print("Carregando o modelo XTTS... (Isso pode demorar um pouco)")
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
    print("✅ Modelo carregado e pronto para ser usado!")
except Exception as e:
    print(f"❌ Erro ao carregar o modelo: {e}")

In [ ]:
# CÉLULA 6: GERAÇÃO, ALINHAMENTO E MONTAGEM

from pydub import AudioSegment
from IPython.display import Audio
import math
import os

def speed_change(sound, speed=1.0):
    """
    Função para alterar a velocidade do áudio via reamostragem (resampling).
    Este método não usa o filtro 'atempo' do FFMPEG e é uma ótima alternativa
    quando o speedup() falha para desaceleração.
    """
    # Cria um novo segmento de áudio com a mesma data raw, mas alterando o frame_rate.
    # Isso "engana" o player para tocar as amostras mais rápido ou mais devagar.
    sound_with_altered_frame_rate = sound._spawn(sound.raw_data, overrides={
        "frame_rate": int(sound.frame_rate * speed)
    })

    # Converte o áudio de volta para o frame_rate padrão.
    # Isso "trava" a mudança de velocidade no áudio, tornando-o compatível com players padrão.
    return sound_with_altered_frame_rate.set_frame_rate(sound.frame_rate)

# --- Preparação para a Montagem ---
# (Certifique-se que a variável 'segmentos_traduzidos' foi criada na Célula 3)
duracao_total_musica_ms = math.ceil(segmentos_traduzidos[-1]['fim']) * 1000
faixa_vocal_final = AudioSegment.silent(duration=duracao_total_musica_ms)

caminho_speaker_wav = caminho_vocal_original # <-- Caminho para sua voz de referência
pasta_segmentos_alinhados = "segmentos_alinhados"
os.makedirs(pasta_segmentos_alinhados, exist_ok=True)

print("\nIniciando geração e alinhamento de cada segmento...")
# --- Loop Principal ---
for i, seg in enumerate(segmentos_traduzidos):
    duracao_original_s = seg['fim'] - seg['inicio']

    if duracao_original_s < 0.2:
        print(f"  Segmento {i} pulado (muito curto).")
        continue

    print(f"Processando segmento {i}/{len(segmentos_traduzidos)}: '{seg['texto_traduzido']}'")

    # 1. Gera o áudio para o texto traduzido
    caminho_temp = os.path.join(pasta_segmentos_alinhados, f"temp_{i}.wav")
    tts.tts_to_file(
        text=seg['texto_traduzido'],
        speaker_wav=caminho_speaker_wav,
        language="pt",
        file_path=caminho_temp
    )

    # 2. Carrega e mede o áudio gerado
    audio_gerado = AudioSegment.from_wav(caminho_temp)
    duracao_gerada_s = len(audio_gerado) / 1000.0
    if duracao_gerada_s == 0: continue # Evita divisão por zero

    # 3. Calcula o fator de ajuste de velocidade
    fator_velocidade = duracao_gerada_s / duracao_original_s

    print(f"  Duração Original: {duracao_original_s:.2f}s | Duração Gerada: {duracao_gerada_s:.2f}s | Fator: {fator_velocidade:.2f}x")

    # 4. Aplica o ajuste de velocidade

    if fator_velocidade < 1.0:
        # Se for para DESACELERAR, usa o método de reamostragem (mais seguro)
        print(f"  -> Desacelerando com o método de reamostragem (speed_change).")
        audio_alinhado = speed_change(audio_gerado, fator_velocidade)

    else:
      # Se for para ACELERAR, usa o método original speedup() (mais rápido e funciona bem)
        audio_alinhado = audio_gerado.speedup(playback_speed=fator_velocidade)

    # 5. Adiciona o segmento na faixa final
    posicao_inicio_ms = seg['inicio'] * 1000
    faixa_vocal_final = faixa_vocal_final.overlay(audio_alinhado, position=posicao_inicio_ms)

print("\nMontagem de todos os segmentos concluída!")

# --- Exportação Final ---
caminho_final_vocal = "vocal_final_traduzido_e_alinhado.wav"
faixa_vocal_final.export(caminho_final_vocal, format="wav")

print(f"\n✅ Faixa vocal final salva em: '{caminho_final_vocal}'")

# Para ouvir no notebook
display(Audio(caminho_final_vocal))

In [ ]:
# CÉLULA 7: JUNÇÃO FINAL (VOCAL + INSTRUMENTAL)

from pydub import AudioSegment
from IPython.display import Audio, display
import os

print("Iniciando a junção da faixa vocal com o instrumental...")

# --- Definição dos Caminhos ---
caminho_vocal_gerado = "/content/vocal_final_traduzido_e_alinhado.wav"
caminho_instrumental = "/content/output/Melodia.wav" # <-- Arquivo gerado pelo Demucs e renomeado
caminho_musica_final = "musica_final_completa.wav"

# --- Verificação de Segurança ---
if not os.path.exists(caminho_vocal_gerado):
    print(f"❌ ERRO: O arquivo do vocal gerado não foi encontrado em '{caminho_vocal_gerado}'")
    print("Por favor, certifique-se de que a Célula 6 foi executada com sucesso.")
elif not os.path.exists(caminho_instrumental):
    print(f"❌ ERRO: O arquivo instrumental não foi encontrado em '{caminho_instrumental}'")
    print("Por favor, certifique-se de que as células do Demucs foram executadas corretamente.")
else:
    # --- Carregamento dos Arquivos ---
    print("Carregando arquivos de áudio...")
    instrumental = AudioSegment.from_wav(caminho_instrumental)
    vocal = AudioSegment.from_wav(caminho_vocal_gerado)

    # --- Ajuste de Volume (MIXAGEM) ---
    # !!! EXPERIMENTE COM ESTE VALOR !!!
    # Use valores positivos para aumentar o volume do vocal, e negativos para diminuir.
    # Ex: +3 (um pouco mais alto), -6 (bem mais baixo), 0 (sem alteração)
    ajuste_de_volume_db = 0
    vocal_ajustado = vocal + ajuste_de_volume_db
    print(f"Ajuste de volume do vocal: {ajuste_de_volume_db} dB")

    # --- Junção (Overlay) ---
    # Sobrepõe o vocal (com volume ajustado) sobre a base instrumental.
    # O Pydub garante que a duração final seja a do arquivo mais longo (o instrumental).
    print("Mixando as faixas...")
    musica_final = instrumental.overlay(vocal_ajustado)

    # --- Exportação do Resultado Final ---
    musica_final.export(caminho_musica_final, format="wav")
    print(f"\n✅ Sucesso! Sua música completa foi salva em: '{caminho_musica_final}'")

    # --- Tocar o áudio no notebook ---
    display(Audio(caminho_musica_final))